# Session 05 · Exact Sampling Distributions

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

## Example 1 - t approaches the normal

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
rng = np.random.default_rng(5)
for df in (2, 5, 30, 1000):
    print(f"df={df:>4}: t(0.975) = {stats.t.ppf(0.975, df):.3f}")
print("normal z(0.975) =", round(stats.norm.ppf(0.975), 3))

df=   2: t(0.975) = 4.303
df=   5: t(0.975) = 2.571
df=  30: t(0.975) = 2.042
df=1000: t(0.975) = 1.962
normal z(0.975) = 1.96


## Example 2 - where chi-square comes from

In [2]:
n, sigma = 8, 5
stat = np.array([(n - 1) * rng.normal(50, sigma, n).var(ddof=1) / sigma ** 2 for _ in range(20000)])
print("mean of simulated statistic:", round(stat.mean(), 2), "(df =", n - 1, ")")
print("simulated 95th pct:", round(np.percentile(stat, 95), 2), "| chi2.ppf:", round(stats.chi2.ppf(0.95, n - 1), 2))

mean of simulated statistic: 7.01 (df = 7 )
simulated 95th pct: 14.08 | chi2.ppf: 14.07


## Example 3 - 95% CI for a variance

In [3]:
fills = rng.normal(500, 3, 15)
n, s2 = len(fills), fills.var(ddof=1)
lo = (n - 1) * s2 / stats.chi2.ppf(0.975, n - 1)
hi = (n - 1) * s2 / stats.chi2.ppf(0.025, n - 1)
print(f"s = {np.sqrt(s2):.2f} ml; 95% CI for sigma: ({np.sqrt(lo):.2f}, {np.sqrt(hi):.2f}) ml")

s = 2.93 ml; 95% CI for sigma: (2.14, 4.62) ml


## Example 4 - goodness-of-fit: is the die fair?

In [4]:
observed = [18, 22, 16, 25, 30, 9]
res = stats.chisquare(observed)
print(res)

Power_divergenceResult(statistic=np.float64(13.5), pvalue=np.float64(0.019117689341227408))


## Example 5 - test of independence

In [5]:
df = pd.DataFrame({"programme": rng.choice(["DS", "Web", "Net"], 300, p=[0.4, 0.4, 0.2])})
p_place = df["programme"].map({"DS": 0.75, "Web": 0.6, "Net": 0.45})
df["placed"] = np.where(rng.random(300) < p_place, "yes", "no")
table = pd.crosstab(df["programme"], df["placed"])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(table, f"\nchi2 = {chi2:.2f}, dof = {dof}, p = {p:.4f}")
print(pd.DataFrame(expected, index=table.index, columns=table.columns).round(1))

placed     no  yes
programme         
DS         24   89
Net        39   38
Web        30   80 
chi2 = 19.65, dof = 2, p = 0.0001
placed       no   yes
programme            
DS         35.0  78.0
Net        23.9  53.1
Web        34.1  75.9


## Example 6 - F-test for equal variances

In [6]:
a, b = rng.normal(60, 5, 20), rng.normal(60, 10, 25)
F = a.var(ddof=1) / b.var(ddof=1)
p = 2 * min(stats.f.sf(F, len(a) - 1, len(b) - 1), stats.f.cdf(F, len(a) - 1, len(b) - 1))
print(f"F = {F:.3f}, p = {p:.4f} | Levene p = {stats.levene(a, b).pvalue:.4f}")

F = 0.214, p = 0.0011 | Levene p = 0.0120
